# AnimeGANv3 — Video to Animation on Colab T4
Select a **T4 GPU**, then **Runtime → Run all**. The notebook clones the benchmark branch, installs dependencies, downloads the Hayao model automatically, asks for one video, renders it, and reports throughput.

In [ ]:
import os, subprocess, sys
REPO = '/content/video-to-animation'
if not os.path.exists(REPO):
    subprocess.run(['git','clone','-b','animeganv3-baseline','https://github.com/v-tech-hub/video-to-animation.git',REPO],check=True)
os.chdir(REPO)
print('Working directory:', os.getcwd())
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip uninstall -y -q onnxruntime onnxruntime-gpu || true
!pip install -q -r requirements-colab.txt


In [ ]:
import sys
import subprocess
import onnxruntime as ort

print('Python:', sys.version)
print('ORT version:', ort.__version__)
print('ORT device:', ort.get_device())
print('Available providers:', ort.get_available_providers())
print('GPU snapshot:')
subprocess.run([
    'nvidia-smi',
    '--query-gpu=name,driver_version,memory.total,memory.used,utilization.gpu',
    '--format=csv,noheader'
], check=False)

assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDAExecutionProvider unavailable.'


In [ ]:
## Benchmark video
By default the notebook asks you to upload your own video (`USE_UPLOAD = True`). Set it to `False` only to use the public sample.


## Benchmark video
By default the notebook downloads a public sample video automatically, so **Run all** needs no upload. Set `USE_UPLOAD = True` in the next cell only when testing your own video.


In [ ]:
# Default reproducible benchmark video from Google's Media CDN.
# Set USE_UPLOAD = True only when you want to test your own video.
USE_UPLOAD = False

from pathlib import Path
import urllib.request
import shutil

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    videos = [n for n in uploaded if Path(n).suffix.lower() in {'.mp4','.mov','.avi','.mkv','.webm'}]
    assert videos, 'Upload a video file.'
    src = Path(videos[0])
    VIDEO = '/content/input' + src.suffix.lower()
    shutil.copyfile(src, VIDEO)
else:
    VIDEO = '/content/input.mp4'
    SAMPLE_URL = 'https://storage.googleapis.com/gtv-videos-bucket/sample/ForBiggerBlazes.mp4'
    if not Path(VIDEO).exists():
        print('Downloading sample benchmark video...')
        urllib.request.urlretrieve(SAMPLE_URL, VIDEO)

print('Video:', VIDEO, f'({Path(VIDEO).stat().st_size / 1024 / 1024:.2f} MB)')


In [ ]:
import os, time, subprocess, threading, cv2
import numpy as np
from PIL import Image
import onnxruntime as ort

BENCH_FRAMES = 120
MAX_EDGE = 1280

cap = cv2.VideoCapture(VIDEO)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
scale = min(1.0, MAX_EDGE / max(w, h))
iw = max(256, int(round(w * scale)) // 8 * 8)
ih = max(256, int(round(h * scale)) // 8 * 8)

frames = []
while len(frames) < BENCH_FRAMES:
    ok, bgr = cap.read()
    if not ok:
        break
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (iw, ih), interpolation=cv2.INTER_AREA)
    frames.append((rgb.astype(np.float32) / 127.5 - 1.0)[None, ...])
cap.release()
assert frames, 'Could not read benchmark frames.'
print(f'Benchmark: {len(frames)} frames at {iw}x{ih}')

def bench(label, providers):
    print(f'\n=== {label} ===', flush=True)
    t0 = time.perf_counter()
    try:
        sess = ort.InferenceSession(MODEL, providers=providers)
    except Exception as e:
        print('SESSION FAILED:', repr(e))
        return None
    init_s = time.perf_counter() - t0
    print('Active providers:', sess.get_providers())
    input_name = sess.get_inputs()[0].name

    # Warm up first; TensorRT engine build/initialization must not pollute steady-state FPS.
    try:
        for f in frames[:min(10, len(frames))]:
            sess.run(None, {input_name: f})
    except Exception as e:
        print('WARMUP FAILED:', repr(e))
        return None

    t0 = time.perf_counter()
    try:
        for f in frames:
            sess.run(None, {input_name: f})
    except Exception as e:
        print('BENCH FAILED:', repr(e))
        return None
    sec = time.perf_counter() - t0
    fps = len(frames) / sec
    print(f'Init/build: {init_s:.2f}s')
    print(f'Steady-state: {sec:.2f}s = {fps:.2f} FPS')
    return fps

results = {}
cuda = bench('CUDA FP32', ['CUDAExecutionProvider'])
if cuda:
    results['gpu'] = cuda

if 'TensorrtExecutionProvider' in ort.get_available_providers():
    trt_options = {
        'device_id': 0,
        'trt_fp16_enable': True,
        'trt_engine_cache_enable': True,
        'trt_engine_cache_path': '/content/trt_cache',
    }
    trt = bench('TensorRT FP16', [('TensorrtExecutionProvider', trt_options), 'CUDAExecutionProvider'])
    if trt:
        results['trt'] = trt
else:
    print('\nTensorRT provider unavailable; skipping TensorRT benchmark.')

assert results, 'No GPU backend benchmark succeeded.'
BEST_DEVICE = max(results, key=results.get)
print('\n=== RESULT ===')
for name, fps in results.items():
    print(f'{name}: {fps:.2f} FPS')
print(f'Winner for full render: {BEST_DEVICE} ({results[BEST_DEVICE]:.2f} FPS)')


In [ ]:
import time, sys, threading
os.chdir(REPO)
os.makedirs('output', exist_ok=True)

# Fail before an expensive render if Colab is still using an old checkout.
script = os.path.join(REPO, 'tools', 'video2anime.py')
code = open(script, 'r').read()
assert 'temp_avi_path' in code and 'codec=MJPEG' in code and 'encoding H.264 MP4' in code, 'Old render code detected; restart/re-clone first.'

cmd = ['python','-u',script,'-i',VIDEO,'-o',os.path.join(REPO,'output'),'-m',MODEL,'-d',BEST_DEVICE]
print('=== Full AnimeGANv3 render ===', flush=True)
print('Backend:', BEST_DEVICE, flush=True)
print('Command:', ' '.join(cmd), flush=True)

stop_gpu = threading.Event()
def gpu_monitor():
    while not stop_gpu.wait(5):
        q = subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,power.draw','--format=csv,noheader,nounits'], capture_output=True, text=True)
        if q.returncode == 0:
            print('[gpu] util%, memMiB, powerW:', q.stdout.strip(), flush=True)
threading.Thread(target=gpu_monitor, daemon=True).start()

start = time.perf_counter()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
returncode = proc.wait()
stop_gpu.set()
elapsed = time.perf_counter() - start
print(f'\nExit code: {returncode}')
print(f'Wall time: {elapsed:.2f} s')
if returncode != 0:
    raise RuntimeError(f'AnimeGANv3 failed with exit code {returncode}')


In [ ]:
import cv2
import subprocess
from pathlib import Path
from google.colab import files

cap = cv2.VideoCapture(VIDEO)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
duration = frames / fps if fps else 0
render_fps = frames / elapsed if elapsed else 0
rtf = elapsed / duration if duration else 0
print(f'Frames: {frames}')
print(f'Source FPS: {fps:.3f}')
print(f'Source duration: {duration:.2f} s')
print(f'Render throughput: {render_fps:.2f} FPS')
print(f'Realtime factor: {rtf:.2f}x')

outs = sorted(Path(REPO, 'output').glob('*.mp4'), key=lambda x: x.stat().st_mtime, reverse=True)
assert outs, 'No rendered MP4 found.'
final_output = outs[0]
probe = subprocess.run([
    'ffprobe','-v','error','-select_streams','v:0',
    '-show_entries','stream=codec_name,width,height,r_frame_rate',
    '-of','default=noprint_wrappers=1',str(final_output)
], text=True, capture_output=True)
print('Output:', final_output)
print(probe.stdout)
assert probe.returncode == 0 and 'codec_name=h264' in probe.stdout, probe.stderr or 'Output is not valid H.264.'
files.download(str(final_output))
